## 03-假设性问题索引（Pre-Question Index）

### 离线构建流程
1. 先将文档分块，对每个块用 LLM 生成若干提问；
2. 将假设性问题向量化，并存入向量数据库；
3. 将每个假设性问题向量与原始块通过 ID 绑定
4. 检索时，将问题向量与假设性问题进行匹配，再取回对应原文块。

#### 对话模型和嵌入模型

In [1]:
import os
from langchain_community.embeddings import DashScopeEmbeddings
embeddings_model = DashScopeEmbeddings(model="text-embedding-v4", dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"))
from langchain_openai import ChatOpenAI
model = ChatOpenAI(api_key=os.getenv("DASHSCOPE_API_KEY"), model="qwen3.7-max", base_url="https://ws-ots0tb33v08803wb.cn-beijing.maas.aliyuncs.com/compatible-mode/v1")

C:\Users\user\AppData\Local\Temp\ipykernel_58104\4281367750.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import DashScopeEmbeddings


In [2]:
# 通过本地ollama上运行的bge-m3做向量化
from langchain_ollama import OllamaEmbeddings
embeddings_model = OllamaEmbeddings(model="bge-m3")

#### 文档加载

In [3]:
from langchain_community.document_loaders import TextLoader
# 加载数据
loader = TextLoader("../../../data/deepseek百度百科.txt", encoding="utf-8")
docs = loader.load()
print(f"文章的长度：{len(docs[0].page_content)}")

文章的长度：20899


#### 文档切块

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=100)
docs = text_splitter.split_documents(docs)
print(f"文档块的数量：{len(docs)}")

文档块的数量：27


#### 用 LLM 为每个块生成多个假设性问题

In [5]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class HypotheticalQuestions(BaseModel):
    """约束生成假设性问题的格式"""
    questions: list[str] = Field(..., description="List of questions")

# 此处使用双括号 {{ 和 }} 是为了在字符串中转义出单个 { 和 }，以确保最终输出的 JSON 格式正确。
prompt = ChatPromptTemplate.from_template(
        """请基于以下文档生成3个假设性问题（必须使用JSON格式）:
        {doc}

        要求：
        1. 输出必须为合法JSON格式，包含questions字段
        2. questions字段的值是包含3个问题的数组
        3. 使用中文提问
        示例格式：
        {{
            "questions": ["问题1", "问题2", "问题3"]
        }}"""
)
# 创建假设性问题链
chain = (
    {"doc": lambda x: x.page_content}
    | prompt
    | model.with_structured_output(HypotheticalQuestions) # 将 LLM 输出构建为字符串列表
    | (lambda x: x.questions) # 提取问题列表
)

"""
    # 其中的 client.with_structured_output 可以理解为输出解析器的一种更高级用法
    将大模型的输出转换为 HypotheticalQuestions 所限定的格式，而 HypotheticalQuestions 要求的格式是：
    定义了一个字段 questions，它具有以下特性：
    - 类型注解：List[str] 表示 questions 字段应该是一个字符串列表。
    - 必需性：Field(...) 中的省略号 ... 表示这个字段是必需的。
    - 描述信息：description="List of questions" 为该字段添加了描述，这对于生成文档或帮助理解模型结构很有用。

    # 测试-在单个文档上调用链，链的最终输出是大模型答复的假设性问题列表
    print("测试：",docs[0])
    print("测试生成问题：",chain.invoke(docs[0]))
"""

'\n    # 其中的 client.with_structured_output 可以理解为输出解析器的一种更高级用法\n    将大模型的输出转换为 HypotheticalQuestions 所限定的格式，而 HypotheticalQuestions 要求的格式是：\n    定义了一个字段 questions，它具有以下特性：\n    - 类型注解：List[str] 表示 questions 字段应该是一个字符串列表。\n    - 必需性：Field(...) 中的省略号 ... 表示这个字段是必需的。\n    - 描述信息：description="List of questions" 为该字段添加了描述，这对于生成文档或帮助理解模型结构很有用。\n\n    # 测试-在单个文档上调用链，链的最终输出是大模型答复的假设性问题列表\n    print("测试：",docs[0])\n    print("测试生成问题：",chain.invoke(docs[0]))\n'

In [6]:
# 批量处理所有文档生成假设性问题（最大并行数5），每个切块后的文档块都对应的生成三个问题
hypothetical_questions = chain.batch(docs, {"max_concurrency": 5})
print("假设性问题列表：",hypothetical_questions)

假设性问题列表： [['假设DeepSeek在研发过程中没有采用数据蒸馏技术，这可能会对其大语言模型的数据处理效率和最终模型性能产生怎样的负面影响？', '如果澳大利亚政府没有以所谓“担心安全风险”为由禁止在政府设备中使用DeepSeek，DeepSeek在海外公共部门的市场拓展可能会迎来哪些新的发展机遇？', '假设DeepSeek-R1模型未能成功登陆英伟达NIM以及被亚马逊和微软接入，这将会对其在国际主流AI生态系统中的普及速度和商业竞争力造成哪些潜在阻碍？'], ['假设幻方量化没有为DeepSeek提供万张A100芯片的强大硬件支持，DeepSeek在早期大语言模型的研发进度和模型性能上可能会受到怎样的影响？', '如果第一财经技术中心在评估后认为DeepSeek V2的架构设计与财经垂类大模型的需求不契合，他们可能会采取哪些替代方案来推进相关研发工作？', '假设DeepSeek在推出V2.5新模型时决定不保持API的向前兼容，这会对现有的API用户和开发者生态造成什么潜在的负面影响？'], ['如果DeepSeek-R1的高效低成本模式被全球其他AI公司广泛采用，英伟达等高端AI芯片制造商的长期市场需求和股价走势可能会发生怎样的根本性变化？', '如果美国政府如经济学家预测的那样，将华芯片出口限制政策全面转向对国内技术的补贴和激励，这会对中美两国在人工智能领域的竞争格局产生什么深远影响？', '如果DeepSeek在未来无法有效解决因用户激增导致的频繁服务宕机问题，这种不稳定性会如何影响其开源模型在全球企业级客户中的商业化落地和长期信任度？'], ['假设DeepSeek在发布Janus-Pro多模态大模型后没有遭遇来自美国IP的大规模恶意攻击，美国白宫将其视为“警钟”并开展国家安全调查的举措是否还会如此迅速地推进？', '如果意大利数据保护机构在20天期限届满后，认为DeepSeek关于个人数据收集和存储的解释不符合其数据保护法规，DeepSeek可能会面临怎样的监管后果？', '假设360集团和中国红客联盟的网络安全支持成功帮助DeepSeek彻底抵御了此次大规模攻击，这是否会促使国内其他人工智能初创公司重新评估并升级其网络安全防御体系？'], ['假设DeepSeek在应对意大利、法国和爱尔兰等欧洲监管机构的审查时，被认定存在严重的数据隐私违规，

In [7]:
print("-------------原始文档--------------")
print(docs[0].page_content)
print("-------------假设性问题--------------")
print(hypothetical_questions[0])

-------------原始文档--------------
DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据。由知名私募巨头幻方量化孕育而生，专注于开发先进的大语言模型（LLM）和相关技术。注册地址：浙江省杭州市拱墅区环城北路169号汇金国际大厦西1幢1201室。法定代表人为裴湉，经营范围包括技术服务、技术开发、软件开发等。
2024年1月5日至6月，相继发布DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2模型。 2024年9月5日，更新API支持文档，宣布合并DeepSeek Coder V2和DeepSeek V2 Chat，推出DeepSeek V2.5。12月13日，发布DeepSeek-VL2。12月26日，正式上线DeepSeek-V3首个版本并同步开源。2025年1月31日，英伟达宣布DeepSeek-R1模型登陆NVIDIANIM。同一时段内，亚马逊和微软也接入DeepSeek-R1模型。 2月5日，DeepSeek-R1、V3、Coder等系列模型，已陆续上线国家超算互联网平台。2月6日消息，澳大利亚政府以所谓“担心安全风险”为由，已禁止在所有政府设备中使用DeepSeek。2月8日，DeepSeek正式登陆苏州，并在苏州市公共算力服务平台上完成部署上线，为用户提供开箱即用的软硬件一体服务。 截至2月9日，DeepSeek App的累计下载量已超1.1亿次，周活跃用户规模最高近9700万。 2月20日消息，已有超200家企业宣布接入DeepSeek。 2月26日，DeepSeek宣布开源DeepGEMM。
-------------假设性问题--------------
['假设DeepSeek在研发过程中没有采用数据蒸馏技术，这可能会对其大语言模型的数据处理效率和最终模型性能产生怎样的负面影响？', '如果澳大利亚政府没有以所谓“担心安全风险”为由禁止在政府设备中使用DeepSeek，DeepSeek在海外公共部门的市场拓展可能会迎来哪些新的发展机遇？', '假设DeepSeek-R1模型未能成功登陆英伟

#### 存储 + 检索
- chromadb 存储 假设性问题向量
- InMemoryByteStore 存储 原始文档

初始化

In [8]:
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryByteStore
# 初始化Chroma实例（用于存储假设性问题向量）
vectorstore = Chroma(collection_name="hypo-questions", embedding_function=embeddings_model)

# 初始化内存字节存储（用于存储原始文档）
store = InMemoryByteStore()

# 初始化多向量检索器（结合向量存储和文档存储）
id_key = "doc_id"
retriever = MultiVectorRetriever(
    vectorstore=vectorstore, # 向量检索库。存的是摘要向量，用来做相似度匹配
    byte_store=store, # 原始文档存储库。存的是原始文档chunk，通过ID关联
    id_key=id_key, # 关联键名。用于将摘要和原始文档进行关联，指定document中元数据中哪个字段作为"桥梁"，把向量库的结果映射回原文
)

将`假设性问题向量`添加到向量数据库（Chroma），将`原始文档块`存储到内存数据库（InMemoryByteStore）

In [9]:
import uuid   # 用于生成唯一标识符
from langchain_core.documents import Document    # 文档对象

# 为每个文档生成唯一ID，该ID用于关联原始文档和假设性问题
doc_ids = [str(uuid.uuid4()) for _ in docs]

# 将假设性问题列表转换为LangChain中Document
question_docs = []
for i, question_list in enumerate(hypothetical_questions):
    question_docs.extend(
        [Document(page_content=s, metadata={id_key: doc_ids[i]}) for s in question_list]
    )

retriever.vectorstore.add_documents(question_docs) # 将问题文档存入向量数据库
retriever.docstore.mset(list(zip(doc_ids, docs))) # 将原始文档存入字节存储（通过ID关联）

In [18]:
# 测试-执行相似性搜索
query = "deepseek受到哪些攻击？"
question_docs = retriever.vectorstore.similarity_search(query)
print("-------------相似性：--------------")
# print('------------假设性问题---------------')
print(question_docs[0].page_content)
ids = [d.metadata[retriever.id_key] for d in question_docs]
_docs = [d for d in retriever.docstore.mget(ids) if d is not None]
print('------------通过假设性问题找到的原始文档---------------')
print(_docs[0].page_content)
print("------------检索器直接通过内部实现-------------")
retrieved_docs = retriever.invoke(query)
print(retrieved_docs[0].page_content)

-------------相似性：--------------
如果与DeepSeek相关的仿冒账号和不实信息在各大社交平台上大规模泛滥，这会对DeepSeek的公众信任度造成什么影响，官方除了发布声明外还能采取哪些进一步的法律或技术维权手段？
------------通过假设性问题找到的原始文档---------------
全面发展
2025年2月5日消息，DeepSeek-R1、V3、Coder等系列模型，已陆续上线国家超算互联网平台。
2025年2月6日，DeepSeek已暂停API服务充值，按钮显示灰色不可用状态。对此官方声明称，“当前服务器资源紧张，为避免对您造成业务影响，我们已暂停API服务充值。同日消息，中国电信、中国移动、中国联通三大运营商相继宣布全面接入DeepSeek。同日，吉利汽车宣布其自研的星睿大模型与DeepSeek R1大模型已完成深度融合，这是汽车行业首次实现此类深度合作。
2025年2月6日，中国外交部发言人郭嘉昆主持例行记者会。有记者提问：据报道，韩国的一些政府部门已屏蔽对DeepSeek的访问，此前意大利、澳大利亚、印度、美国、日本等国也传出禁止或限制使用DeepSeek的消息，一些企业也屏蔽对DeepSeek访问。中方如何看待上述动向？郭嘉昆：我注意到有关情况。我想强调的是，中国政府高度重视并依法保护数据隐私和安全，从来没有，也不会要求企业或个人以违法的形式采集或存储数据。中方一贯反对泛化国家安全概念、将经贸科技问题政治化的做法，同时中方也将坚定维护中国企业的合法权益。
2025年2月6日，路透社最新消息，澳大利亚政府以所谓“担心安全风险”为由，已禁止在所有政府设备中使用DeepSeek。
2025年2月6日，外交部发言人郭嘉昆主持例行记者会。有记者提问，据报道，韩国的一些政府部门已经屏蔽对DeepSeek的访问，之前意大利、澳大利亚、印度、美国、日本等国家也传出禁止或限制使用DeepSeek的消息，一些企业也屏蔽对DeepSeek访问。中方如何看待这些动向？郭嘉昆表示，中国政府高度重视并依法保护数据隐私和安全，从来没有也不会要求企业或个人以违法的形式采集或存储数据。中方一贯反对泛化国家安全概念、将经贸科技问题政治化的做法，同时中方也将坚定维护中国企业的合法权益。
2025年2月6日，援引韩国官员的说法称，韩国国防部也已经屏蔽

#### 问题回答

增强+生成

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate

prompt1 =  ChatPromptTemplate.from_template("根据下面的文档回答问题:\n\n{doc}\n\n问题: {question}")

# 生成问题回答链
chain = RunnableParallel({
    "doc": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
}) | prompt1 | model | StrOutputParser()

# 生成问题回答
answer = chain.invoke({"question": query})
print("-------------回答--------------")
print(answer)
#  返回的是知识块
retrieved_docs = retriever.invoke(query)
print("-------------检索到的问题--------------")
print(retrieved_docs)

-------------回答--------------
根据提供的文档内容，DeepSeek主要受到了以下两方面的“攻击”与恶意行为：

**1. 大规模恶意网络攻击（技术层面）**
* **攻击事件**：2025年1月27日晚至1月28日凌晨，DeepSeek线上服务遭到大规模恶意攻击。为维持服务，官方一度暂时限制了+86手机号以外的注册方式（后修改提示为“注册可能繁忙，请稍等重试”）。
* **攻击来源**：据奇安信安全专家分析，DeepSeek此次受到的网络攻击，其IP地址全部来自美国。

**2. 商标恶意抢注（知识产权与商业层面）**
* **恶意抢注**：2025年2月，随着DeepSeek在全球引发广泛关注，个别企业和自然人恶意“蹭热点”，向国家知识产权局提交了“DEEPSEEK”等相关商标的注册申请，意图牟取不当利益。
* **官方打击**：国家知识产权局坚决打击此类恶意申请行为，已依法对63件相关商标注册申请予以驳回。

*(注：除了上述直接攻击外，文档还提及DeepSeek在海外市场遭遇了多国政府、军队及监管机构的**行政限制与打压**，如美国海军和国会众议院禁用、多国政府部门屏蔽访问，以及美、意、法、爱尔兰等国监管机构对其发起的国家安全审查或数据隐私调查等。)*
-------------检索到的问题--------------
[Document(metadata={'source': '../../data/deepseek百度百科.txt'}, page_content='全面发展\n2025年2月5日消息，DeepSeek-R1、V3、Coder等系列模型，已陆续上线国家超算互联网平台。\n2025年2月6日，DeepSeek已暂停API服务充值，按钮显示灰色不可用状态。对此官方声明称，“当前服务器资源紧张，为避免对您造成业务影响，我们已暂停API服务充值。同日消息，中国电信、中国移动、中国联通三大运营商相继宣布全面接入DeepSeek。同日，吉利汽车宣布其自研的星睿大模型与DeepSeek R1大模型已完成深度融合，这是汽车行业首次实现此类深度合作。\n2025年2月6日，中国外交部发言人郭嘉昆主持例行记者会。有记者提问：据报道，韩国的一些政府部门已屏蔽对DeepSeek的访问，此前意大利、澳大利亚、印度、美国、日本等国也传出禁止